# TRAIN MODEL INDOBERT BASE

In [13]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset
import pandas as pd

In [14]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Menggunakan GPU (CUDA) 🚀")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Menggunakan Mac GPU (MPS) 🍏")
else:
    device = torch.device("cpu")
    print("Menggunakan CPU 🐢")

Menggunakan Mac GPU (MPS) 🍏


In [15]:
df_bersih = pd.read_csv('dataset_judi_clean.csv')
df_bersih = df_bersih.dropna(subset=['komentar_super_clean', 'label'])

teks = df_bersih['komentar_super_clean'].tolist()
label = df_bersih['label'].tolist()

In [16]:
train_teks, test_teks, train_label, test_label = train_test_split(
    teks, label, test_size=0.2, random_state=42, stratify=label
)

pd.DataFrame({
    'komentar_super_clean': test_teks,
    'label': test_label
}).to_csv('test_set.csv', index=False)

print("Test set disimpan ke test_set.csv")

Test set disimpan ke test_set.csv


In [17]:
nama_model = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(nama_model)

train_encodings = tokenizer(train_teks, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_teks, truncation=True, padding=True, max_length=128)

class DatasetJudi(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = DatasetJudi(train_encodings, train_label)
test_dataset = DatasetJudi(test_encodings, test_label)

In [18]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
import torch.nn as nn
import numpy as np

In [19]:
model = AutoModelForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p1", num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 155.16it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc, 
        'f1': f1, 
        'precision': precision, 
        'recall': recall
        }

In [21]:
bobot_kelas = compute_class_weight('balanced', classes=np.unique(train_label), y=train_label)
bobot_tensor = torch.tensor(bobot_kelas, dtype=torch.float).to(device)

In [22]:
class CustomWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=bobot_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [23]:
training_args = TrainingArguments(
    output_dir='./hasil_model_indobert_base_v3', 
    num_train_epochs=5,                  
    per_device_train_batch_size=16,      
    per_device_eval_batch_size=16,
    eval_strategy="epoch",               
    save_strategy="epoch",               
    learning_rate=2e-5,        
    weight_decay=0.01,
    load_best_model_at_end=True,         
    metric_for_best_model="f1",          
)

In [24]:
trainer = CustomWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("MEMULAI TRAINING INDOBERT BASE")
trainer.train()

MEMULAI TRAINING INDOBERT BASE


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.104235,0.056408,0.989736,0.987418,0.996372,0.978622
2,0.031861,0.044610,0.990225,0.988124,0.988124,0.988124
3,0.014331,0.052606,0.992180,0.990476,0.992840,0.988124
4,0.004749,0.059254,0.992669,0.991045,0.996399,0.985748
5,0.003610,0.051618,0.991691,0.989887,0.991657,0.988124


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shard

TrainOutput(global_step=2560, training_loss=0.031232698005624115, metrics={'train_runtime': 3765.2639, 'train_samples_per_second': 10.866, 'train_steps_per_second': 0.68, 'total_flos': 1240519806588300.0, 'train_loss': 0.031232698005624115, 'epoch': 5.0})

In [25]:
trainer.save_model('./indobert_base_judol_final')
tokenizer.save_pretrained('./indobert_base_judol_final')

print("\nTRAINING BASE SELESAI!")

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.76s/it]


TRAINING BASE SELESAI!


# Evaluation Model

In [26]:
hasil_evaluasi = trainer.evaluate()

print("\n--- EVALUASI INDOBERT BASE --- ")
print(f"Akurasi (Accuracy)   : {hasil_evaluasi['eval_accuracy'] * 100:.2f}%")
print(f"F1-Score             : {hasil_evaluasi['eval_f1'] * 100:.2f}%")
print(f"Presisi (Precision)  : {hasil_evaluasi['eval_precision'] * 100:.2f}%")
print(f"Recall               : {hasil_evaluasi['eval_recall'] * 100:.2f}%")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



--- EVALUASI INDOBERT BASE --- 
Akurasi (Accuracy)   : 99.27%
F1-Score             : 99.10%
Presisi (Precision)  : 99.64%
Recall               : 98.57%


# UJI COBA MODEL

In [27]:
from transformers import pipeline
import unicodedata
import re

def bersihkan_obfuscation(teks):
    if not isinstance(teks, str): return ""
    teks = unicodedata.normalize('NFKC', teks)
    hasil = []
    for char in teks:
        try:
            nama = unicodedata.name(char)
            if 'SQUARED LATIN' in nama or 'REGIONAL INDICATOR SYMBOL' in nama:
                hasil.append(nama.split()[-1].lower())
            elif 'GREEK CAPITAL LETTER LAMDA' in nama or 'GREEK SMALL LETTER LAMDA' in nama:
                hasil.append('a')
            else:
                hasil.append(char)
        except ValueError:
            if char == '@': hasil.append('a')
            else: hasil.append(char)
    teks_baru = "".join(hasil).lower()
    teks_baru = re.sub(r'[^a-z0-9\s]', ' ', teks_baru)
    return re.sub(r'\s+', ' ', teks_baru).strip()

print("Memanggil IndoBERT Base yang sudah dilatih...")
prediktor = pipeline(
    "text-classification", 
    model="./indobert_base_judol_final", 
    tokenizer="./indobert_base_judol_final"
)

komentar_tes = [
    "Bang, tutornya mantap banget sangat membantu buat ngerjain tugas kampus!",
    
    "Ayo daftar sekarang di situs slot gacor terpercaya pasti wd paus",
    
    "wuihh info g4c1r malam ini bosku, mampir ke 🅿🆄🅻🅰🆄🆆🅸🅽 auto sultan!!",
    "modal receh m4in di MΛNDALIKΛ77 pasti banjir scatter ✌✌",
    
    "Build item hero ini emang paling gacor bang, damage-nya nembus armor musuh."
]

print("\n--- HASIL INTEROGASI INDOBERT BASE ---")
for komen in komentar_tes:
    komen_bersih = bersihkan_obfuscation(komen)
    hasil = prediktor(komen_bersih)[0]
    
    # Terjemahkan Label (0 = Normal, 1 = Judi)
    if hasil['label'] == 'LABEL_1':
        prediksi = "🔴 JUDI ONLINE"
    else:
        prediksi = "🟢 NORMAL"
        
    keyakinan = hasil['score'] * 100
    
    print(f"Komentar Asli : '{komen}'")
    print(f"Prediksi      : {prediksi} (Tingkat Keyakinan: {keyakinan:.1f}%)\n")
    print("-" * 60)

Memanggil IndoBERT Base yang sudah dilatih...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1444.80it/s, Materializing param=classifier.weight]                                      



--- HASIL INTEROGASI INDOBERT BASE ---
Komentar Asli : 'Bang, tutornya mantap banget sangat membantu buat ngerjain tugas kampus!'
Prediksi      : 🟢 NORMAL (Tingkat Keyakinan: 100.0%)

------------------------------------------------------------
Komentar Asli : 'Ayo daftar sekarang di situs slot gacor terpercaya pasti wd paus'
Prediksi      : 🔴 JUDI ONLINE (Tingkat Keyakinan: 100.0%)

------------------------------------------------------------
Komentar Asli : 'wuihh info g4c1r malam ini bosku, mampir ke 🅿🆄🅻🅰🆄🆆🅸🅽 auto sultan!!'
Prediksi      : 🔴 JUDI ONLINE (Tingkat Keyakinan: 100.0%)

------------------------------------------------------------
Komentar Asli : 'modal receh m4in di MΛNDALIKΛ77 pasti banjir scatter ✌✌'
Prediksi      : 🔴 JUDI ONLINE (Tingkat Keyakinan: 100.0%)

------------------------------------------------------------
Komentar Asli : 'Build item hero ini emang paling gacor bang, damage-nya nembus armor musuh.'
Prediksi      : 🟢 NORMAL (Tingkat Keyakinan: 100.0%)

----